In [0]:
# ===============================================
# Notebook: Load dim_brand from staging
# ===============================================

from pyspark.sql.functions import col

# --------------------------------------
# 1. Leitura da tabela de staging
# --------------------------------------
df_sales_raw = spark.table("beverage_analytics.staging.abi_bus_case1_beverage_sales_20210726")

# --------------------------------------
# 2. Seleção e transformação
# --------------------------------------
dim_brand = df_sales_raw.select("CE_BRAND_FLVR", "BRAND_NM") \
    .withColumn("CE_BRAND_FLVR", col("CE_BRAND_FLVR").cast("int")) \
    .dropDuplicates()
dim_brand = dim_brand.toDF(*[c.lower() for c in dim_brand.columns])

# --------------------------------------
# 3. Escrita no Unity Catalog (Delta Table)
# --------------------------------------
catalog = "beverage_analytics"
schema = "dim"
table = "dim_brand"


dim_brand.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{catalog}.{schema}.{table}")

print(f"Carga concluída com sucesso em: {catalog}.{schema}.{table}")
